# La búsqueda del techo — el cuaderno que la corre

Este cuaderno **corre** la búsqueda de techos y nada más. No arma ninguna tabla ni conclusión — eso vive en `Benchmark_Search_Report_v1.ipynb`, que lee el registro que esto deja y lo presenta. Es la misma división que ya tienen la campaña y su informe, y por la misma razón: re-renderizar el informe cuesta segundos porque no vuelve a buscar nada.

Existía la mitad que dibuja y no la que corre: la única forma de pedir esta búsqueda era el paso `search-pilot` llamando a `harness.run_search()` por su cuenta, así que el artefacto que se abre y se lee nunca ejercitaba el programa que produce lo que muestra.

Se llamaba `Benchmark_Search_Pilot_v1`. El nombre afirmaba una escala que este cuaderno no tiene por qué elegir, y mientras la afirmaba **ningún cuaderno podía correr la búsqueda completa**: a escala completa los techos salían de la función de biblioteca y todo lo demás del recorrido salía de un cuaderno, que es la misma divergencia ---el ensayo ejercita una cosa y la corrida real hace otra--- que esta reestructura existe para cerrar, un nivel más abajo. `Benchmark_Search_v1` no se recicla: un nombre liberado que vuelve deja que una referencia vieja siga resolviendo, contra otro artefacto.

> **La escala la recibe, no la elige: es un modo del recorrido entero.**
>
> `config.is_pilot_scale()`, igual que la campaña, el barrido y el diagnóstico. En ensayo escribe `ceilings.pilot.json` y a escala completa `ceilings.json`, y en las dos el resto del recorrido lee el archivo de *su* escala: en ensayo todo corre y consume lo del ensayo, y a escala completa nada del ensayo se usa.
>
> **Acá estaba fijo en `True`.** El argumento escrito era que una sola respuesta elige las dos cosas ---a qué escala corre y a qué archivo escribe--- y que derivarla dejaría que el tamaño de la CAMPAÑA decidiera si este cuaderno lanza la corrida larga. La primera mitad es correcta y sigue siendo la razón por la que hay UNA respuesta y no dos. La segunda costaba más de lo que compraba: con `True` fijo, **ningún cuaderno podía correr la búsqueda a escala completa**, así que a escala completa los techos venían de la función de biblioteca mientras todo lo demás venía de un cuaderno. La autorización sigue existiendo y está donde corresponde ---en el paso `search-pilot`, que se niega cuando la escala configurada no es la del ensayo, exactamente como `campaign-local`, `noise-sweep` y `noise-diagnostic`---, así que abrir este cuaderno a mano a escala completa es una decisión que alguien toma; la forja no la toma sola.
>
> A escala de ensayo su respuesta no se cita. La rampa avanza con la fracción de entrenamiento transcurrida, así que a escala corta satura en la segunda época y todo techo se alcanza casi enseguida: lo que mediría es un paisaje en el que la campaña no entrena nunca. Lo que contesta es *«¿el programa corre?»*, que es lo único que un ensayo puede contestar --- y el registro lo dice de sí mismo con `atRequiredScale`, que es lo que hace que una campaña real se niegue a consumirlo.

In [ ]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


In [ ]:
# The repository was resolved by the cell above -- the forge's own, travelling
# byte for byte -- and this one only uses it. `REPOSITORY` is still the name
# every cell below reads, so nothing below changes.
import os
import sys
from pathlib import Path

REPOSITORY = ROOT
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

In [ ]:
from MIL_CREDA_Benchmark import config, harness

# La escala, recibida y con nombre. La misma respuesta elige las dos cosas --- a
# qué escala corre la búsqueda y a qué archivo escribe ---, así que escribirla en
# los dos lugares serían dos ortografías de una decisión, y la que quede vieja se
# lee igual de verde que la que no.
#
# `config.is_pilot_scale()` y no un `True` fijo: la escala es un modo del
# recorrido entero y no una elección de cada paso, y es la misma lectura que
# usan la campaña, el barrido y el diagnóstico --- las dos constantes que
# separan una escala de la otra son `EPOCHS` y `SEEDS`, leídas una sola vez y en
# un solo lugar. Con `True` fijo acá, ningún cuaderno podía correr la búsqueda
# completa, y a escala completa los techos salían de la biblioteca mientras todo
# el resto del recorrido salía de un cuaderno.
#
# La autorización para gastar la corrida larga no vive acá sino en el paso que
# ejecuta este cuaderno, que se niega cuando la escala configurada no es la del
# ensayo --- la misma guarda que ya tenían `campaign-local`, `noise-sweep` y
# `noise-diagnostic`, y por la misma razón: sus `produces` nombran el árbol de
# ensayo y ninguno más.
ES_ENSAYO = config.is_pilot_scale()
DESTINO = config.ceilings_record_for(ES_ENSAYO)

print("escala:", "ensayo" if ES_ENSAYO else "completa")
print("escribe en:", DESTINO)
print("registro en vigor ahora mismo:", config.ceilings_provenance()["source"])

## Lo que cuesta

El aforo declarado, y ningún pronóstico cronometrado. La campaña paga una corrida
real antes de comprometerse con la rejilla entera porque un estimado del costo es
más barato que el costo; acá el ensayo COMPLETO es del orden de esa corrida, así
que cronometrar una para pronosticarlo costaría una fracción visible de lo que
pronostica.

Se imprimen las dos: la escala declarada de la búsqueda y la que esta ejecución
va a correr. A escala completa son la misma, y verlas juntas es lo que dice cuál
de las dos está pasando antes de que empiece a gastar.

In [ ]:
aforo = config.search_sizing()
print(f"motor: {aforo['engine']}")
print(f"la búsqueda a su escala declarada: {aforo['trials']} trials de "
      f"{aforo['epochs']} épocas por (familia, transferencia) — "
      f"{aforo['runs']} corridas")
# Y lo que va a correr ESTA ejecución, que es lo de arriba a escala completa y
# la escala propia del ensayo si no. Las dos constantes salen de `config` y no
# de una aritmética escrita acá: el motor elige entre ellas con el mismo
# `pilot`, y un segundo cálculo sería la ortografía que queda vieja.
trials = config.PILOT_SEARCH_TRIALS if ES_ENSAYO else aforo["trials"]
epocas = config.PILOT_SEARCH_EPOCHS if ES_ENSAYO else aforo["epochs"]
print(f"esta corrida ({'ensayo' if ES_ENSAYO else 'completa'}): {trials} trials "
      f"de {epocas} épocas — "
      f"{trials * aforo['families'] * aforo['transfers']} corridas")

## La corrida

Se busca una vez. Si el registro ya existe se lee y no se vuelve a buscar: que el
registro exista significa que la búsqueda contestó, y sobrescribir una respuesta
porque alguien quería otra es el refinanciamiento silencioso que la negativa de la
campaña existe para impedir. Para volver a empezar hay que borrarlo a mano.

In [ ]:
# La biblioteca computa y este cuaderno orquesta: `run_search` arma la reducción
# a la escala que le corresponde a la búsqueda, despacha al motor que
# `config.SEARCH_ENGINE` declara y relee del disco lo que quedó escrito. Nada de
# eso se vuelve a escribir acá.
registro = harness.run_search(pilot=ES_ENSAYO)

print("registro escrito en:", DESTINO)
print("familias con techo:", ", ".join(sorted(registro)))
print()
print("las tablas y la conclusión las arma Benchmark_Search_Report_v1.ipynb")